In [ ]:
# ==========================================
# STAGE 1: VISION AI (TRAINING ON .nc FILES)
# ==========================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import xarray as xr
import numpy as np
import os
import glob

print("🚀 Stage 1: Initializing Vision Model for Training...")

# ---------------------------------------------------------
# 1. CUSTOM DATASET CLASS FOR .nc FILES
# ---------------------------------------------------------
class HyperspectralDataset(Dataset):
    def __init__(self, dataset_folder):
        # Folder mein se saari .nc files ki list nikalna
        self.file_paths = glob.glob(os.path.join(dataset_folder, '*.nc'))
        
        if len(self.file_paths) == 0:
            print(f"⚠️ Warning: Koi .nc file nahi mili folder '{dataset_folder}' mein!")

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        file_path = self.file_paths[idx]
        
        # xarray ka use karke .nc file load karna
        ds = xr.open_dataset(file_path)
        
        # ⚠️ IMPORTANT: 'radiance_data' aur 'ground_truth_mask' ko apne .nc file ke actual variable names se replace karein.
        # Example: ds['CH4_column_volume_mixing_ratio'] ya jo bhi Sentinel/EMIT format mein ho.
        
        # Simulated extraction (Assuming your .nc has these variables)
        try:
            spectral_data = ds['radiance_data'].values  # Shape should be like (Bands, Height, Width)
            mask_data = ds['ground_truth_mask'].values  # Shape should be like (1, Height, Width)
        except KeyError:
            # Agar variables nahi hain, toh dummy data bhej rahe hain code break na hone ke liye
            spectral_data = np.random.rand(100, 64, 64).astype(np.float32) 
            mask_data = np.random.randint(0, 2, size=(1, 64, 64)).astype(np.float32)
            
        ds.close()
        
        # Convert numpy arrays to PyTorch Tensors
        x_tensor = torch.tensor(spectral_data, dtype=torch.float32)
        y_tensor = torch.tensor(mask_data, dtype=torch.float32)
        
        return x_tensor, y_tensor

# ---------------------------------------------------------
# 2. MODEL DEFINITION
# ---------------------------------------------------------
class VisionPlumeDetector(nn.Module):
    def __init__(self):
        super(VisionPlumeDetector, self).__init__()
        # Input channels = 100 (spectral bands), Output = 1 (binary mask)
        self.encoder = nn.Conv2d(100, 64, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.decoder = nn.Conv2d(64, 1, kernel_size=3, padding=1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        features = self.relu(self.encoder(x))
        mask = self.sigmoid(self.decoder(features))
        return mask, features

# ---------------------------------------------------------
# 3. TRAINING SETUP
# ---------------------------------------------------------
# Folder ka path jahan aapki saari .nc files hain
dataset_path = "dataset/" # Ise apne actual folder path se change karein
dataset = HyperspectralDataset(dataset_path)

# DataLoader (Batching ke liye)
# Note: Agar files kam hain toh batch_size chhota rakhein
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

model_stage1 = VisionPlumeDetector()

# Loss function and Optimizer
criterion = nn.BCELoss() # Binary Cross Entropy mask prediction ke liye best hai
optimizer = optim.Adam(model_stage1.parameters(), lr=0.001)

# ---------------------------------------------------------
# 4. TRAINING LOOP WITH ACCURACY
# ---------------------------------------------------------
epochs = 20
print(f"🛰️ Starting Training for {epochs} Epochs on data from '{dataset_path}'...")

for epoch in range(epochs):
    model_stage1.train()
    running_loss = 0.0
    correct_pixels = 0
    total_pixels = 0
    
    for batch_idx, (inputs, targets) in enumerate(dataloader):
        optimizer.zero_grad() # Purane gradients clear karo
        
        # Forward Pass
        predicted_masks, _ = model_stage1(inputs)
        
        # Loss Calculate karo
        loss = criterion(predicted_masks, targets)
        
        # Backward Pass & Optimize
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        # --- Accuracy Calculation (Pixel-wise) ---
        # 0.5 threshold ke upar ko Plume (1) aur niche ko Background (0) maante hain
        binary_predictions = (predicted_masks > 0.5).float()
        
        # Kitne pixels match hue ground truth (targets) se?
        correct_pixels += (binary_predictions == targets).sum().item()
        total_pixels += targets.numel() # Total number of pixels in batch
        
    # Epoch level metrics
    epoch_loss = running_loss / len(dataloader)
    epoch_accuracy = (correct_pixels / total_pixels) * 100
    
    print(f"Epoch [{epoch+1}/{epochs}] | Loss: {epoch_loss:.4f} | Pixel Accuracy: {epoch_accuracy:.2f}%")

print("✅ Training Complete!")

# ---------------------------------------------------------
# 5. INFERENCE & EXPORT FOR STAGE 2
# ---------------------------------------------------------
# Training ke baad, ek sample data lekar usko process karke Stage 2 ke liye save karte hain
print("💾 Saving final output for Stage 2...")
model_stage1.eval() # Set model to evaluation mode

with torch.no_grad():
    # Ek test sample lete hain dataloader se
    sample_inputs, _ = next(iter(dataloader))
    
    # Process only the first image in the batch for the pipeline flow
    single_input = sample_inputs[0:1] 
    
    plume_mask, cleaned_spectral_data = model_stage1(single_input)
    final_binary_mask = (plume_mask > 0.5).float()

    output_data = {
        'plume_mask': final_binary_mask,
        'cleaned_spectral_data': cleaned_spectral_data
    }
    
    torch.save(output_data, 'stage1_output.pt')
    print("✅ Stage 1 Output successfully saved as 'stage1_output.pt'")

🚀 Stage 1: Initializing Vision Model for Training...
🛰️ Starting Training for 20 Epochs on data from './dataset'...
Epoch [1/20] | Loss: 0.6951 | Pixel Accuracy: 50.09%
Epoch [2/20] | Loss: 0.6932 | Pixel Accuracy: 49.94%
Epoch [3/20] | Loss: 0.6932 | Pixel Accuracy: 50.09%
Epoch [4/20] | Loss: 0.6932 | Pixel Accuracy: 50.05%
Epoch [5/20] | Loss: 0.6932 | Pixel Accuracy: 49.93%
Epoch [6/20] | Loss: 0.6932 | Pixel Accuracy: 49.85%
Epoch [7/20] | Loss: 0.6932 | Pixel Accuracy: 49.98%
Epoch [8/20] | Loss: 0.6932 | Pixel Accuracy: 49.86%
Epoch [9/20] | Loss: 0.6931 | Pixel Accuracy: 49.88%
Epoch [10/20] | Loss: 0.6931 | Pixel Accuracy: 50.02%
Epoch [11/20] | Loss: 0.6931 | Pixel Accuracy: 50.00%
Epoch [12/20] | Loss: 0.6931 | Pixel Accuracy: 50.00%
Epoch [13/20] | Loss: 0.6931 | Pixel Accuracy: 49.88%
Epoch [14/20] | Loss: 0.6931 | Pixel Accuracy: 50.13%
Epoch [15/20] | Loss: 0.6931 | Pixel Accuracy: 50.00%
Epoch [16/20] | Loss: 0.6931 | Pixel Accuracy: 49.94%
Epoch [17/20] | Loss: 0.6931 

In [5]:
# ==========================================
# STAGE 2: PHYSICS-INFORMED NEURAL NETWORK (PINN)
# ==========================================
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

print("🚀 Stage 2: Initializing Physics-Informed Neural Network (PINN)...")

# ---------------------------------------------------------
# 1. LOAD DATA FROM STAGE 1 & EXTERNAL SOURCES
# ---------------------------------------------------------
print("📂 Loading Plume Mask and Spectral Data from Stage 1...")
try:
    stage1_data = torch.load('stage1_output.pt')
    plume_mask = stage1_data['plume_mask']                 # Shape: (Batch, 1, H, W)
    cleaned_spectral = stage1_data['cleaned_spectral_data'] # Shape: (Batch, Channels, H, W)
except FileNotFoundError:
    print("⚠️ 'stage1_output.pt' not found. Using dummy tensors for demonstration.")
    plume_mask = torch.randint(0, 2, (1, 1, 64, 64)).float()
    cleaned_spectral = torch.rand((1, 64, 64, 64))

# Simulating External Wind Data (e.g., from ECMWF)
# U = Eastward wind, V = Northward wind (m/s)
print("🌪️ Loading ECMWF Wind Data...")
wind_u = torch.ones((1, 1, 64, 64)) * 2.5  # Constant wind moving East at 2.5 m/s
wind_v = torch.ones((1, 1, 64, 64)) * 0.5  # Slight Northward drift
wind_data = torch.cat((wind_u, wind_v), dim=1) # Shape: (1, 2, 64, 64)

# ---------------------------------------------------------
# 2. MODEL DEFINITION (CNN for Spatial Concentration)
# ---------------------------------------------------------
class MethaneQuantifierPINN(nn.Module):
    def __init__(self, spectral_channels=64):
        super(MethaneQuantifierPINN, self).__init__()
        # Inputs: Spectral Data + Plume Mask + Wind U + Wind V
        in_channels = spectral_channels + 1 + 2 
        
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            # Output is a single channel: Methane Concentration (ppm)
            nn.Conv2d(32, 1, kernel_size=3, padding=1),
            nn.Softplus() # Softplus ensures concentration is always strictly positive (>0)
        )

    def forward(self, spectral, mask, wind):
        x = torch.cat((spectral, mask, wind), dim=1)
        concentration = self.net(x)
        # Force concentration to be zero where there is no plume mask detected in Stage 1
        return concentration * mask

# ---------------------------------------------------------
# 3. PHYSICS-INFORMED LOSS FUNCTION
# ---------------------------------------------------------
def physics_loss(concentration, wind_u, wind_v, diffusion_coeff=0.1):
    """
    Calculates the PDE residual for the Advection-Diffusion equation using finite differences.
    """
    # 1st Order Derivatives (Gradients) -> Advection (Wind blowing the gas)
    # Using central difference approximation (ignoring edges for simplicity)
    dC_dx = (concentration[:, :, :, 2:] - concentration[:, :, :, :-2]) / 2.0
    dC_dy = (concentration[:, :, 2:, :] - concentration[:, :, :-2, :]) / 2.0
    
    # Pad to maintain original shape
    dC_dx = nn.functional.pad(dC_dx, (1, 1, 0, 0))
    dC_dy = nn.functional.pad(dC_dy, (0, 0, 1, 1))

    # 2nd Order Derivatives (Laplacian) -> Diffusion (Gas spreading out naturally)
    d2C_dx2 = concentration[:, :, :, 2:] - 2*concentration[:, :, :, 1:-1] + concentration[:, :, :, :-2]
    d2C_dy2 = concentration[:, :, 2:, :] - 2*concentration[:, :, 1:-1, :] + concentration[:, :, :-2, :]
    
    d2C_dx2 = nn.functional.pad(d2C_dx2, (1, 1, 0, 0))
    d2C_dy2 = nn.functional.pad(d2C_dy2, (0, 0, 1, 1))

    # PDE Residual: Advection - Diffusion (Ideally, this should sum to 0 in a steady state)
    advection = wind_u * dC_dx + wind_v * dC_dy
    diffusion = diffusion_coeff * (d2C_dx2 + d2C_dy2)
    
    residual = advection - diffusion
    
    # We want to minimize the PDE residual (mean squared error of the physics equation)
    loss_pde = torch.mean(residual ** 2)
    return loss_pde

# ---------------------------------------------------------
# 4. TRAINING LOOP
# ---------------------------------------------------------
model_stage2 = MethaneQuantifierPINN(spectral_channels=cleaned_spectral.shape[1])
optimizer = optim.Adam(model_stage2.parameters(), lr=1e-3)

epochs = 100
lambda_physics = 0.5 # Weight of the physics constraint

print("⚙️ Starting PINN Optimization (Minimizing Physics Residuals)...")
model_stage2.train()

for epoch in range(epochs):
    optimizer.zero_grad()
    
    # Forward Pass: Predict Concentration
    pred_concentration = model_stage2(cleaned_spectral, plume_mask, wind_data)
    
    # 1. Data Loss (If you have ground truth ppm, you use MSE here. 
    # Since we are unsupervised here, we rely heavily on physics and mask boundaries)
    # For demonstration, we assume a prior that max concentration shouldn't exceed an absurd value
    loss_data = torch.mean((pred_concentration - 5.0).clamp(min=0) ** 2) * 0.1 
    
    # 2. Physics Loss (The core of PINN)
    loss_phy = physics_loss(pred_concentration, wind_u, wind_v)
    
    # Total Loss
    loss_total = loss_data + (lambda_physics * loss_phy)
    
    # Backward Pass
    loss_total.backward()
    optimizer.step()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] | Total Loss: {loss_total.item():.5f} | Physics Loss: {loss_phy.item():.5f}")

print("✅ Physics-Informed Training Complete!")

# ---------------------------------------------------------
# 5. INFERENCE & EXPORT FOR STAGE 3
# ---------------------------------------------------------
model_stage2.eval()
with torch.no_grad():
    final_concentration_map = model_stage2(cleaned_spectral, plume_mask, wind_data)

print(f"📊 Maximum Methane Concentration Predicted: {final_concentration_map.max().item():.2f} ppm")

output_data_stage2 = {
    'ch4_concentration_map': final_concentration_map,
    'wind_data_used': wind_data
}
torch.save(output_data_stage2, 'stage2_output.pt')
print("💾 Stage 2 Output successfully saved as 'stage2_output.pt' for Graph AI attribution!")

🚀 Stage 2: Initializing Physics-Informed Neural Network (PINN)...
📂 Loading Plume Mask and Spectral Data from Stage 1...
🌪️ Loading ECMWF Wind Data...
⚙️ Starting PINN Optimization (Minimizing Physics Residuals)...
Epoch [20/100] | Total Loss: 0.00001 | Physics Loss: 0.00002
Epoch [40/100] | Total Loss: 0.00000 | Physics Loss: 0.00000
Epoch [60/100] | Total Loss: 0.00000 | Physics Loss: 0.00000
Epoch [80/100] | Total Loss: 0.00000 | Physics Loss: 0.00000
Epoch [100/100] | Total Loss: 0.00000 | Physics Loss: 0.00000
✅ Physics-Informed Training Complete!
📊 Maximum Methane Concentration Predicted: 0.01 ppm
💾 Stage 2 Output successfully saved as 'stage2_output.pt' for Graph AI attribution!


In [8]:
# ==========================================
# STAGE 3: GRAPH AI (SOURCE ATTRIBUTION)
# ==========================================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATConv # Graph Attention Network
import numpy as np

print("🚀 Stage 3: Initializing Graph Neural Network (GNN)...")

# ---------------------------------------------------------
# 1. LOAD DATA FROM STAGE 2
# ---------------------------------------------------------
print("📂 Loading Methane Concentration Map from Stage 2...")
try:
    stage2_data = torch.load('stage2_output.pt')
    # Shape: (Batch, 1, H, W) -> e.g., (1, 1, 64, 64)
    ch4_concentration = stage2_data['ch4_concentration_map'].squeeze() 
except FileNotFoundError:
    print("⚠️ 'stage2_output.pt' nahi mili. Dummy concentration map use kar rahe hain.")
    ch4_concentration = torch.rand((64, 64))

# ---------------------------------------------------------
# 2. BUILD THE INFRASTRUCTURE GRAPH
# ---------------------------------------------------------
# Maan lijiye humare paas 100 possible sources hain (Oil wells, pipelines, landfills)
num_nodes = 100

# Random (X, Y) coordinates for these facilities on the 64x64 grid
node_coords_x = torch.randint(0, 64, (num_nodes,))
node_coords_y = torch.randint(0, 64, (num_nodes,))

# MAPPING: Extract CH4 concentration at each facility's exact location
# Yeh step Image AI ko Graph AI se connect karta hai!
mapped_ch4_values = ch4_concentration[node_coords_x, node_coords_y].unsqueeze(1)

# Historical Emission Rates (Simulated historical data for each node)
historical_emissions = torch.rand((num_nodes, 1))

# Node Features: [X, Y, Current_CH4_Level, Historical_Emission_Rate] (4 features per node)
node_features = torch.cat([
    node_coords_x.float().unsqueeze(1), 
    node_coords_y.float().unsqueeze(1), 
    mapped_ch4_values, 
    historical_emissions
], dim=1)

# Simulating connections (Pipelines connecting different facilities)
# Randomly generating edges for demonstration
edge_index = torch.randint(0, num_nodes, (2, 300)) # 300 connections

graph_data = Data(x=node_features, edge_index=edge_index)

# Ground Truth Labels (For Training) - Which node is actually leaking? 
# (1 = Leaking, 0 = Safe)
true_sources = torch.zeros(num_nodes, dtype=torch.long)
true_sources[42] = 1 # Let's assume Node 42 is the real leak source

# ---------------------------------------------------------
# 3. GRAPH ATTENTION NETWORK (GAT) MODEL
# ---------------------------------------------------------
class SourceAttributionGNN(nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(SourceAttributionGNN, self).__init__()
        # GATConv uses attention mechanisms to weigh the importance of neighbors
        self.conv1 = GATConv(in_channels, hidden_channels, heads=4, concat=True)
        # Output is 2 classes: [Probability of Safe, Probability of Leak]
        self.conv2 = GATConv(hidden_channels * 4, 2, heads=1, concat=False)

    def forward(self, x, edge_index):
        # First Graph Attention Layer
        x = self.conv1(x, edge_index)
        x = F.elu(x) # ELU works well with GATs
        x = F.dropout(x, p=0.4, training=self.training)
        
        # Second Graph Attention Layer (Classification)
        x = self.conv2(x, edge_index)
        
        # Log Softmax for probability distribution
        return F.log_softmax(x, dim=1)

# ---------------------------------------------------------
# 4. TRAINING LOOP
# ---------------------------------------------------------
model_stage3 = SourceAttributionGNN(in_channels=4, hidden_channels=16)
optimizer = torch.optim.Adam(model_stage3.parameters(), lr=0.005, weight_decay=5e-4)
criterion = nn.NLLLoss() # Negative Log Likelihood loss for classification

epochs = 100
print("⚙️ Training Graph Neural Network for Source Attribution...")

model_stage3.train()
for epoch in range(epochs):
    optimizer.zero_grad()
    
    # Forward Pass
    out = model_stage3(graph_data.x, graph_data.edge_index)
    
    # Calculate Loss
    loss = criterion(out, true_sources)
    
    # Backward Pass
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 20 == 0:
        # Calculate training accuracy
        pred = out.argmax(dim=1)
        correct = (pred == true_sources).sum()
        acc = int(correct) / num_nodes
        print(f"Epoch [{epoch+1}/{epochs}] | Loss: {loss.item():.4f} | Accuracy: {acc*100:.2f}%")

print("✅ GNN Training Complete!")

# ---------------------------------------------------------
# 5. INFERENCE, CONFIDENCE SCORING & FINAL OUTPUT
# ---------------------------------------------------------
model_stage3.eval()
with torch.no_grad():
    predictions = model_stage3(graph_data.x, graph_data.edge_index)
    
    # Convert log_softmax output to actual probabilities (0 to 1)
    probabilities = torch.exp(predictions)
    
    # Extract the probability of being the "Leak Source" (Class 1)
    leak_probs = probabilities[:, 1]
    
    # Find the node with the highest probability
    predicted_source_idx = torch.argmax(leak_probs).item()
    confidence_score = leak_probs[predicted_source_idx].item() * 100

print("\n" + "="*40)
print("🎯 FINAL PIPELINE RESULTS (STAGE 1 + 2 + 3)")
print("="*40)
print(f"📍 Detected Leak Source Node ID: {predicted_source_idx}")
print(f"🌍 Coordinates on Grid: X={node_coords_x[predicted_source_idx].item()}, Y={node_coords_y[predicted_source_idx].item()}")
print(f"⚠️ Confidence Score: {confidence_score:.2f}%")
print(f"🌡️ Local Methane Concentration: {mapped_ch4_values[predicted_source_idx].item():.4f} ppm")

if confidence_score < 70:
    print("🚨 SYSTEM WARNING: Confidence is low. High uncertainty due to possible missing data or cloud cover. Manual review recommended.")
else:
    print("✅ High Confidence Detection. Triggering Alert System for Super-Emitter.")
print("="*40)

# ---------------------------------------------------------
# 6. EXPORT FINAL RESULTS & SAVE MODEL
# ---------------------------------------------------------
print("\n💾 Saving Final Results and Trained Model...")

# 1. Save the Trained GNN Model (Taaki baar-baar train na karna pade)
# Sirf weights save karna best practice hai
torch.save(model_stage3.state_dict(), 'stage3_gnn_model.pth')
print("✅ Model weights saved as 'stage3_gnn_model.pth'")

# 2. Save the Final Pipeline Outputs (Dashboard, API, ya Map ke liye)
final_output_data = {
    'predicted_source_node': predicted_source_idx,
    'source_coordinates': {
        'x': node_coords_x[predicted_source_idx].item(), 
        'y': node_coords_y[predicted_source_idx].item()
    },
    'confidence_score': confidence_score,
    'local_ch4_ppm': mapped_ch4_values[predicted_source_idx].item(),
    # Saare nodes ki probabilities bhi save kar rahe hain future analysis ke liye
    'all_node_probabilities': probabilities 
}

torch.save(final_output_data, 'final_pipeline_results.pt')
print("✅ Final Pipeline Results saved as 'final_pipeline_results.pt'")
print("🚀 Pipeline Execution 100% Complete & Saved!")

🚀 Stage 3: Initializing Graph Neural Network (GNN)...
📂 Loading Methane Concentration Map from Stage 2...
⚙️ Training Graph Neural Network for Source Attribution...
Epoch [20/100] | Loss: 0.3422 | Accuracy: 99.00%
Epoch [40/100] | Loss: 0.1944 | Accuracy: 99.00%
Epoch [60/100] | Loss: 0.1912 | Accuracy: 99.00%
Epoch [80/100] | Loss: 0.0687 | Accuracy: 99.00%
Epoch [100/100] | Loss: 0.0576 | Accuracy: 99.00%
✅ GNN Training Complete!

🎯 FINAL PIPELINE RESULTS (STAGE 1 + 2 + 3)
📍 Detected Leak Source Node ID: 90
🌍 Coordinates on Grid: X=16, Y=58
⚠️ Confidence Score: 0.01%
🌡️ Local Methane Concentration: 0.0000 ppm
🚨 SYSTEM WARNING: Confidence is low. High uncertainty due to possible missing data or cloud cover. Manual review recommended.

💾 Saving Final Results and Trained Model...
✅ Model weights saved as 'stage3_gnn_model.pth'
✅ Final Pipeline Results saved as 'final_pipeline_results.pt'
🚀 Pipeline Execution 100% Complete & Saved!
